In [1]:
import pandas as pd
import icartt
import os
import warnings
import re
from datetime import datetime
import csv
from datetime import datetime, timedelta
from netCDF4 import Dataset
import numpy as np
from scipy import stats
import glob
from math import pi
import ast

In [ ]:
bins = [3.1, 3, 4, 5, 6]
data_strats = [1, 2]

#Set which "MAC_binning_{binned_number}bins_optimized" file to use
binned_number = 3.1
data_strat = 1

STEP 0: METHODS 1 - 4 as functions

In [2]:
# Refactored Data Splitting Methods as Functions

def method1_kfold_split(df, df_kfold_info, output_path, model_columns):
    """Method 1: Nested Cross-Validation (5-Fold Outer + 3-Fold Inner)"""
    import os
    import pandas as pd
    import random
    from itertools import combinations
    
    FINAL_OUT_PATH = os.path.join(output_path, "kfold_based_split")
    os.makedirs(FINAL_OUT_PATH, exist_ok=True)
    
    print("Starting Nested Cross-Validation dataset splitting...")
    print(f"Original dataset shape: {df.shape}")
    
    # Get all unique campaigns in the dataset for reference
    all_campaigns_in_data = set(df['Campaign'].unique())
    print(f"All campaigns in dataset: {sorted(all_campaigns_in_data)}")
    
    # Filter function
    def filter_invalid_data(df):
        """Filter out rows with NaN, empty values, -2222, or -9999 in model_columns"""
        if df.shape[0] == 0:
            return df
        
        mask = True
        for col in model_columns:
            if col in df.columns:
                col_mask = (~df[col].isna()) & (df[col] != -2222) & (df[col] != -9999) & (df[col] != '')
                mask = mask & col_mask
            else:
                print(f"Warning: Column '{col}' not found in dataset")
        return df[mask].copy()
    
    # ============================================================================
    # STEP 1: Create Hold-out Evaluation Set
    # ============================================================================
    print("\\n" + "="*60)
    print("STEP 1: CREATING HOLD-OUT EVALUATION SET")
    print("="*60)
    
    # Get evaluation campaigns from CSV
    evaluation_campaigns_str = df_kfold_info['evaluation_campaigns'].iloc[0]
    evaluation_campaigns = [camp.strip() for camp in evaluation_campaigns_str.split(',') if camp.strip()]
    
    print(f"Evaluation campaigns (hold-out): {evaluation_campaigns}")
    
    # Create evaluation dataset
    evaluation_df = df[df['Campaign'].isin(evaluation_campaigns)].copy()
    print(f"Evaluation data shape (before filtering): {evaluation_df.shape}")
    
    # Filter evaluation data
    evaluation_df_filtered = filter_invalid_data(evaluation_df)
    print(f"Evaluation data shape (after filtering): {evaluation_df_filtered.shape}")
    print(f"Evaluation rows removed: {evaluation_df.shape[0] - evaluation_df_filtered.shape[0]:,}")
    
    # Save evaluation set
    evaluation_dir = os.path.join(FINAL_OUT_PATH, "evaluation")
    os.makedirs(evaluation_dir, exist_ok=True)
    evaluation_path = os.path.join(evaluation_dir, "evaluation.csv")
    evaluation_df_filtered.to_csv(evaluation_path, index=False)
    print(f"Saved evaluation.csv to: {evaluation_path}")
    
    # ============================================================================
    # STEP 2: Get Remaining Campaigns for Nested CV
    # ============================================================================
    print("\\n" + "="*60)
    print("STEP 2: PREPARING CAMPAIGNS FOR NESTED CV")
    print("="*60)
    
    # Get remaining campaigns (exclude evaluation campaigns)
    remaining_campaigns = list(all_campaigns_in_data - set(evaluation_campaigns))
    print(f"Remaining campaigns for nested CV: {sorted(remaining_campaigns)}")
    print(f"Number of campaigns for nested CV: {len(remaining_campaigns)}")
    
    if len(remaining_campaigns) < 5:
        print(f"ERROR: Need at least 5 campaigns for 5-fold CV, but only have {len(remaining_campaigns)}")
        raise ValueError("Insufficient campaigns for 5-fold cross-validation")
    
    # ============================================================================
    # STEP 3: Create Outer Folds (5-Fold) with Constraint
    # ============================================================================
    print("\\n" + "="*60)
    print("STEP 3: CREATING OUTER FOLDS (5-FOLD)")
    print("="*60)
    
    random.seed(42)  # For reproducibility
    
    def create_outer_folds_with_constraint(campaigns, n_folds=5):
        """Create outer folds ensuring all campaigns appear in at least one test fold and aiming for 80:20 train:test ratio"""
        
        # Get row counts for each campaign (filtered data)
        campaign_rows = {}
        for campaign in campaigns:
            campaign_data = df[df['Campaign'] == campaign]
            campaign_rows[campaign] = filter_invalid_data(campaign_data).shape[0]
        
        total_rows = sum(campaign_rows.values())
        target_test_rows_per_fold = total_rows * 0.2 / n_folds  # 20% of total divided by number of folds
        
        print(f"Total rows in {len(campaigns)} campaigns: {total_rows:,}")
        print(f"Target test rows per fold: {target_test_rows_per_fold:,.0f}")
        print(f"Campaign row counts: {campaign_rows}")
        
        # Sort campaigns by row count for better distribution
        sorted_campaigns = sorted(campaigns, key=lambda x: campaign_rows[x], reverse=True)
        
        folds = []
        campaigns_used_in_test = set()
        
        for fold_idx in range(n_folds):
            print(f"\\n  Creating fold {fold_idx + 1}...")
            
            # For the last fold, ensure all unused campaigns are included
            if fold_idx == n_folds - 1:
                unused_campaigns = set(campaigns) - campaigns_used_in_test
                if unused_campaigns:
                    test_campaigns = list(unused_campaigns)
                    print(f"    Last fold - including all unused campaigns: {test_campaigns}")
                else:
                    # If all campaigns used, create balanced split from remaining options
                    test_campaigns = find_best_test_combination(campaigns, campaign_rows, target_test_rows_per_fold, campaigns_used_in_test)
            else:
                # Find best combination for this fold
                test_campaigns = find_best_test_combination(campaigns, campaign_rows, target_test_rows_per_fold, campaigns_used_in_test)
            
            train_campaigns = [c for c in campaigns if c not in test_campaigns]
            
            # Calculate actual ratios
            test_rows = sum(campaign_rows[c] for c in test_campaigns)
            train_rows = sum(campaign_rows[c] for c in train_campaigns)
            total_fold_rows = train_rows + test_rows
            
            test_pct = (test_rows / total_fold_rows * 100) if total_fold_rows > 0 else 0
            train_pct = (train_rows / total_fold_rows * 100) if total_fold_rows > 0 else 0
            
            print(f"    Test campaigns: {test_campaigns} ({test_rows:,} rows, {test_pct:.1f}%)")
            print(f"    Train campaigns: {train_campaigns} ({train_rows:,} rows, {train_pct:.1f}%)")
            
            folds.append({
                'fold': fold_idx + 1,
                'train_campaigns': train_campaigns,
                'test_campaigns': test_campaigns
            })
            
            campaigns_used_in_test.update(test_campaigns)
        
        return folds
    
    def find_best_test_combination(all_campaigns, campaign_rows, target_test_rows, used_campaigns):
        """Find the best combination of campaigns for test set to achieve target ratio"""
        available_campaigns = [c for c in all_campaigns if c not in used_campaigns]
        
        # If no unused campaigns, allow reuse but prefer unused ones
        if not available_campaigns:
            available_campaigns = all_campaigns
            print(f"    No unused campaigns - allowing reuse from: {available_campaigns}")
        else:
            print(f"    Available unused campaigns: {available_campaigns}")
        
        best_combination = []
        best_score = float('inf')
        
        max_campaigns_to_try = min(len(available_campaigns), 4)  # Limit combinations for performance
        
        for r in range(1, max_campaigns_to_try + 1):
            for combo in combinations(available_campaigns, r):
                combo_rows = sum(campaign_rows[c] for c in combo)
                # Score based on how close we are to target (prefer slightly under to over)
                score = abs(combo_rows - target_test_rows)
                if combo_rows > target_test_rows:
                    score *= 1.2  # Penalize going over target
                
                if score < best_score:
                    best_score = score
                    best_combination = list(combo)
            
            # If we found a decent combination, don't try larger combinations
            if best_combination and best_score < target_test_rows * 0.3:  # Within 30% of target
                break
        
        # If no good combination found, just take the campaign closest to target
        if not best_combination:
            best_combination = [min(available_campaigns, key=lambda x: abs(campaign_rows[x] - target_test_rows))]
        
        return best_combination
    
    # Create outer folds
    outer_folds = create_outer_folds_with_constraint(remaining_campaigns, n_folds=5)
    
    print("Outer fold configuration:")
    for fold in outer_folds:
        print(f"  Fold {fold['fold']}: Train={len(fold['train_campaigns'])}, Test={len(fold['test_campaigns'])}")
        print(f"    Train campaigns: {fold['train_campaigns']}")
        print(f"    Test campaigns: {fold['test_campaigns']}")
    
    # Process and save folds (simplified version - full implementation would include inner folds)
    for outer_fold in outer_folds:
        fold_num = outer_fold['fold']
        fold_dir = os.path.join(FINAL_OUT_PATH, f"kfold{fold_num}")
        os.makedirs(fold_dir, exist_ok=True)
        
        # Create outer fold datasets
        outer_train_df = df[df['Campaign'].isin(outer_fold['train_campaigns'])].copy()
        outer_test_df = df[df['Campaign'].isin(outer_fold['test_campaigns'])].copy()
        
        # Filter outer fold data
        outer_train_df_filtered = filter_invalid_data(outer_train_df)
        outer_test_df_filtered = filter_invalid_data(outer_test_df)
        
        # Save outer fold data
        outer_train_path = os.path.join(fold_dir, "train.csv")
        outer_test_path = os.path.join(fold_dir, "test.csv")
        
        outer_train_df_filtered.to_csv(outer_train_path, index=False)
        outer_test_df_filtered.to_csv(outer_test_path, index=False)
    
    print(f"K-fold split completed: {FINAL_OUT_PATH}")


def method2_campaign_split(df, df_campaign_based_info, output_path, model_columns):
    """Method 2: Campaign-based Split (Train/Validation/Evaluation)"""
    import os
    import pandas as pd
    
    FINAL_OUT_PATH = os.path.join(output_path, "campaign_based_split")
    os.makedirs(FINAL_OUT_PATH, exist_ok=True)
    
    print("Starting Campaign-based dataset splitting...")
    print(f"Original dataset shape: {df.shape}")
    
    print("\\n" + "="*60)
    print("METHOD 2: Splitting by Entire Campaigns (Train/Validation/Evaluation)")
    print("="*60)
    
    # Get train, validation, and evaluation campaign lists from the CSV
    train_campaigns_str = df_campaign_based_info['train_campaigns'].iloc[0]
    validation_campaigns_str = df_campaign_based_info['validation_campaigns'].iloc[0]
    evaluation_campaigns_str = df_campaign_based_info['evaluation_campaigns'].iloc[0]
    
    # Convert comma-separated strings to lists
    train_campaigns = [camp.strip() for camp in train_campaigns_str.split(',')]
    validation_campaigns = [camp.strip() for camp in validation_campaigns_str.split(',')]
    evaluation_campaigns = [camp.strip() for camp in evaluation_campaigns_str.split(',')]
    
    print(f"Train campaigns: {train_campaigns}")
    print(f"Validation campaigns: {validation_campaigns}")
    print(f"Evaluation campaigns: {evaluation_campaigns}")
    
    # Split the dataset by campaigns
    train_df = df[df['Campaign'].isin(train_campaigns)].copy()
    validation_df = df[df['Campaign'].isin(validation_campaigns)].copy()
    evaluation_df = df[df['Campaign'].isin(evaluation_campaigns)].copy()
    
    print(f"Train set shape: {train_df.shape}")
    print(f"Validation set shape: {validation_df.shape}")
    print(f"Evaluation set shape: {evaluation_df.shape}")
    
    # Filter out rows with NaN, empty values, -2222, or -9999 in model_columns
    def filter_invalid_data(df):
        mask = True
        for col in model_columns:
            if col in df.columns:
                col_mask = (~df[col].isna()) & (df[col] != -2222) & (df[col] != -9999) & (df[col] != '')
                mask = mask & col_mask
            else:
                print(f"Warning: Column '{col}' not found in dataset")
        return df[mask].copy()
    
    train_df_filtered = filter_invalid_data(train_df)
    validation_df_filtered = filter_invalid_data(validation_df)
    evaluation_df_filtered = filter_invalid_data(evaluation_df)
    
    print(f"After filtering - Train: {train_df_filtered.shape[0]:,} rows, Validation: {validation_df_filtered.shape[0]:,} rows, Evaluation: {evaluation_df_filtered.shape[0]:,} rows")
    
    # Save results
    train_path = os.path.join(FINAL_OUT_PATH, "train.csv")
    validation_path = os.path.join(FINAL_OUT_PATH, "validation.csv")
    evaluation_path = os.path.join(FINAL_OUT_PATH, "evaluation.csv")
    
    train_df_filtered.to_csv(train_path, index=False)
    validation_df_filtered.to_csv(validation_path, index=False)
    evaluation_df_filtered.to_csv(evaluation_path, index=False)
    
    print(f"Campaign-based split completed: {FINAL_OUT_PATH}")


def method3_day_split(df, df_day_based_info, output_path, model_columns):
    """Method 3: Day-based Split with Hold-out Campaigns"""
    import os
    import pandas as pd
    
    FINAL_OUT_PATH = os.path.join(output_path, "day_based_split")
    os.makedirs(FINAL_OUT_PATH, exist_ok=True)
    
    print("Starting Day-based dataset splitting...")
    print(f"Original dataset shape: {df.shape}")
    
    print("\\n" + "="*60)
    print("METHOD 3: Splitting by Dates with Hold-out Campaigns (Train/Test/Evaluation)")
    print("="*60)
    
    # Check if hold_out column exists, if not create it with default value 0
    if 'hold_out' not in df_day_based_info.columns:
        print("WARNING: 'hold_out' column not found. Creating with default value 0 (no hold-out campaigns)")
        df_day_based_info['hold_out'] = 0
    
    # Separate hold-out campaigns from date-split campaigns
    holdout_campaigns = df_day_based_info[df_day_based_info['hold_out'] == 1]['campaign'].tolist()
    date_split_campaigns = df_day_based_info[df_day_based_info['hold_out'] == 0]
    
    print(f"Hold-out campaigns (entire campaigns go to evaluation): {holdout_campaigns}")
    print(f"Date-split campaigns (split by specific dates): {date_split_campaigns['campaign'].tolist()}")
    
    # Initialize datasets
    train_df_list = []
    test_df_list = []
    evaluation_df_list = []
    
    # Process hold-out campaigns first
    if holdout_campaigns:
        print(f"\\nProcessing hold-out campaigns...")
        for campaign in holdout_campaigns:
            campaign_data = df[df['Campaign'] == campaign].copy()
            if campaign_data.shape[0] > 0:
                evaluation_df_list.append(campaign_data)
                print(f"  {campaign}: {campaign_data.shape[0]:,} rows → evaluation.csv")
    
    # Helper function to safely parse date strings
    def parse_dates_safely(date_str):
        """Safely parse date string, handling NaN and various formats"""
        if pd.isna(date_str):
            return []
        
        if not isinstance(date_str, str):
            date_str = str(date_str)
        
        if date_str.strip().lower() in ['nan', 'none', '']:
            return []
        
        dates = [date.strip() for date in date_str.split(',') if date.strip()]
        return dates
    
    # Process date-split campaigns
    print(f"\\nProcessing date-split campaigns...")
    for _, row in date_split_campaigns.iterrows():
        campaign = row['campaign']
        train_dates_str = row['train_dates']
        test_dates_str = row['test_dates']
        
        print(f"\\n--- Processing Campaign: {campaign} ---")
        
        # Get campaign data
        campaign_data = df[df['Campaign'] == campaign].copy()
        if campaign_data.shape[0] == 0:
            print(f"  WARNING: No data found for campaign '{campaign}'")
            continue
        
        # Parse train and test dates safely
        train_dates = parse_dates_safely(train_dates_str)
        test_dates = parse_dates_safely(test_dates_str)
        
        # Convert Date column to string for consistent comparison
        campaign_data['Date'] = campaign_data['Date'].astype(str)
        
        # Get train data for this campaign
        if train_dates:
            campaign_train_data = campaign_data[campaign_data['Date'].isin(train_dates)].copy()
            if campaign_train_data.shape[0] > 0:
                train_df_list.append(campaign_train_data)
                print(f"  Train data: {campaign_train_data.shape[0]:,} rows")
        
        # Get test data for this campaign
        if test_dates:
            campaign_test_data = campaign_data[campaign_data['Date'].isin(test_dates)].copy()
            if campaign_test_data.shape[0] > 0:
                test_df_list.append(campaign_test_data)
                print(f"  Test data: {campaign_test_data.shape[0]:,} rows")
    
    # Combine all dataframes
    train_df = pd.concat(train_df_list, ignore_index=True) if train_df_list else pd.DataFrame()
    test_df = pd.concat(test_df_list, ignore_index=True) if test_df_list else pd.DataFrame()
    evaluation_df = pd.concat(evaluation_df_list, ignore_index=True) if evaluation_df_list else pd.DataFrame()
    
    # Filter invalid data
    def filter_invalid_data(df):
        if df.shape[0] == 0:
            return df
        
        mask = True
        for col in model_columns:
            if col in df.columns:
                col_mask = (~df[col].isna()) & (df[col] != -2222) & (df[col] != -9999) & (df[col] != '')
                mask = mask & col_mask
        return df[mask].copy()
    
    train_df_filtered = filter_invalid_data(train_df)
    test_df_filtered = filter_invalid_data(test_df)
    evaluation_df_filtered = filter_invalid_data(evaluation_df)
    
    # Save results
    train_path = os.path.join(FINAL_OUT_PATH, "train.csv")
    test_path = os.path.join(FINAL_OUT_PATH, "test.csv")
    evaluation_path = os.path.join(FINAL_OUT_PATH, "evaluation.csv")
    
    train_df_filtered.to_csv(train_path, index=False)
    test_df_filtered.to_csv(test_path, index=False)
    evaluation_df_filtered.to_csv(evaluation_path, index=False)
    
    print(f"Day-based split completed: {FINAL_OUT_PATH}")


def method4_random_split(df, df_random_based_info, output_path, model_columns):
    """Method 4: Random-based Split (80:20 Train/Test + Hold-out Evaluation)"""
    import os
    import pandas as pd
    from sklearn.model_selection import train_test_split
    
    FINAL_OUT_PATH = os.path.join(output_path, "random_based_split")
    os.makedirs(FINAL_OUT_PATH, exist_ok=True)
    
    print("Starting Random-based dataset splitting...")
    print(f"Original dataset shape: {df.shape}")
    
    print("\\n" + "="*60)
    print("METHOD 4: Random-Based Split (80:20 Train/Test + Hold-out Evaluation)")
    print("="*60)
    
    # Get held-out campaigns
    held_out_str = df_random_based_info['held_out'].iloc[0]
    held_out_campaigns = [camp.strip() for camp in held_out_str.split(',') if camp.strip()]
    
    print(f"Held-out campaigns (go to evaluation): {held_out_campaigns}")
    
    # Split data into held-out (evaluation) and remaining (for random split)
    evaluation_df = df[df['Campaign'].isin(held_out_campaigns)].copy()
    remaining_df = df[~df['Campaign'].isin(held_out_campaigns)].copy()
    
    print(f"Evaluation data (held-out campaigns): {evaluation_df.shape[0]:,} rows")
    print(f"Remaining data (for random split): {remaining_df.shape[0]:,} rows")
    
    # Random 80:20 split of remaining data
    print(f"\\nPerforming random 80:20 split on remaining {remaining_df.shape[0]:,} rows...")
    
    if remaining_df.shape[0] > 0:
        train_df, test_df = train_test_split(
            remaining_df, 
            test_size=0.2, 
            random_state=42,  # For reproducibility
            shuffle=True
        )
        
        print(f"Random split results:")
        print(f"  Train: {train_df.shape[0]:,} rows ({100*train_df.shape[0]/remaining_df.shape[0]:.1f}%)")
        print(f"  Test: {test_df.shape[0]:,} rows ({100*test_df.shape[0]/remaining_df.shape[0]:.1f}%)")
    else:
        print("ERROR: No remaining data for random split!")
        train_df = pd.DataFrame()
        test_df = pd.DataFrame()
    
    # Filter invalid data
    def filter_invalid_data(df):
        if df.shape[0] == 0:
            return df
        
        mask = True
        for col in model_columns:
            if col in df.columns:
                col_mask = (~df[col].isna()) & (df[col] != -2222) & (df[col] != -9999) & (df[col] != '')
                mask = mask & col_mask
        return df[mask].copy()
    
    train_df_filtered = filter_invalid_data(train_df)
    test_df_filtered = filter_invalid_data(test_df)
    evaluation_df_filtered = filter_invalid_data(evaluation_df)
    
    # Save results
    train_path = os.path.join(FINAL_OUT_PATH, "train.csv")
    test_path = os.path.join(FINAL_OUT_PATH, "test.csv")
    evaluation_path = os.path.join(FINAL_OUT_PATH, "evaluation.csv")
    
    train_df_filtered.to_csv(train_path, index=False)
    test_df_filtered.to_csv(test_path, index=False)
    evaluation_df_filtered.to_csv(evaluation_path, index=False)
    
    print(f"Random-based split completed: {FINAL_OUT_PATH}")

STEP 1: Run for all combinations of bins and data_strats

In [ ]:
# Master Batch Processing Cell - Runs All Methods for All Combinations

# Configuration
bins = [3.1, 3, 4, 5, 6]
data_strats = [1, 2]

print("="*100)
print("STARTING BATCH PROCESSING OF ALL DATA SPLITTING METHODS")
print("="*100)
print(f"Processing {len(bins)} bins × {len(data_strats)} data strategies = {len(bins) * len(data_strats)} total combinations")
print(f"Running 4 methods per combination = {len(bins) * len(data_strats) * 4} total method executions")
print("="*100)

# Loop through all combinations
for i, binned_number in enumerate(bins):
    for j, data_strat in enumerate(data_strats):
        combination_num = i * len(data_strats) + j + 1
        total_combinations = len(bins) * len(data_strats)
        
        print(f"\\n{'='*100}")
        print(f"COMBINATION {combination_num}/{total_combinations}: bins={binned_number}, data_strat={data_strat}")
        print(f"{'='*100}")
        
        try:
            # ================================================================
            # SETUP: Load dataset and configure paths for this combination
            # ================================================================
            print(f"\\n{'-'*50}")
            print(f"SETUP: Loading dataset and configuring paths...")
            print(f"{'-'*50}")
            
            # READ ENTIRE DATASET
            if binned_number == 3.1:
                dataset_path = rf"C:\\Users\\haika\\Desktop\\May_Research\\MAC Machine Learning Model Code\\MAC_DATASET_LLOD_FILTERED.csv"
            else:
                dataset_path = rf"C:\\Users\\haika\\Desktop\\May_Research\\MAC Machine Learning Model Code\\binning_datasets\\MAC_binning_{binned_number}bins_optimized.csv"
            
            df = pd.read_csv(dataset_path)
            print(f"Dataset loaded: {df.shape}")
            
            # Define output path
            if binned_number == 3.1:
                output_path = rf"C:\\Users\\haika\\Desktop\\May_Research\\MAC Machine Learning Model Code\\split_datasets\\data_split_strat{data_strat}\\MAC_DATASET_LLOD_FILTERED"
            else:
                output_path = rf"C:\\Users\\haika\\Desktop\\May_Research\\MAC Machine Learning Model Code\\split_datasets\\data_split_strat{data_strat}\\MAC_binning_{binned_number}bins_optimized"
            
            print(f"Output path: {output_path}")
            
            # Define info_lists
            kfold_info_path = rf"C:\\Users\\haika\\Desktop\\May_Research\\MAC Machine Learning Model Code\\info_lists\\kfold_info_list{data_strat}.csv"
            campaign_based_info_path = rf"C:\\Users\\haika\\Desktop\\May_Research\\MAC Machine Learning Model Code\\info_lists\\campaign_based_info_list{data_strat}.csv"
            day_based_info_path = rf"C:\\Users\\haika\\Desktop\\May_Research\\MAC Machine Learning Model Code\\info_lists\\day_based_info_list{data_strat}.csv"
            random_based_info_path = rf"C:\\Users\\haika\\Desktop\\May_Research\\MAC Machine Learning Model Code\\info_lists\\random_based_info_list{data_strat}.csv"
            
            # Read them as dfs
            df_kfold_info = pd.read_csv(kfold_info_path)
            df_campaign_based_info = pd.read_csv(campaign_based_info_path)
            df_day_based_info = pd.read_csv(day_based_info_path)
            df_random_based_info = pd.read_csv(random_based_info_path)
            
            print(f"Info lists loaded")
            
            # Columns important for model runs
            model_columns = ['N1', 'N2', 'N3', 'N_total', 'V1', 'V2', 'V3', 'V_total', 'SAE', 'AAE', 'SSA_red', 'SSA_blue', 'SSA_green', 'b_scat_red', 'b_scat_blue', 'b_scat_green', 'b_abs_red', 'b_abs_blue', 'b_abs_green', 'MAC_bc']
            
            # ================================================================
            # METHOD 1: K-fold Cross-Validation
            # ================================================================
            print(f"\\n{'-'*50}")
            print(f"METHOD 1: K-fold Cross-Validation")
            print(f"{'-'*50}")
            
            method1_kfold_split(df, df_kfold_info, output_path, model_columns)
            
            # ================================================================
            # METHOD 2: Campaign-based Split
            # ================================================================
            print(f"\\n{'-'*50}")
            print(f"METHOD 2: Campaign-based Split")
            print(f"{'-'*50}")
            
            method2_campaign_split(df, df_campaign_based_info, output_path, model_columns)
            
            # ================================================================
            # METHOD 3: Day-based Split
            # ================================================================
            print(f"\\n{'-'*50}")
            print(f"METHOD 3: Day-based Split")
            print(f"{'-'*50}")
            
            method3_day_split(df, df_day_based_info, output_path, model_columns)
            
            # ================================================================
            # METHOD 4: Random-based Split
            # ================================================================
            print(f"\\n{'-'*50}")
            print(f"METHOD 4: Random-based Split")
            print(f"{'-'*50}")
            
            method4_random_split(df, df_random_based_info, output_path, model_columns)
            
            # ================================================================
            # COMBINATION COMPLETED
            # ================================================================
            print(f"\\n{'SUCCESS '*10}")
            print(f"COMBINATION {combination_num}/{total_combinations} COMPLETED SUCCESSFULLY!")
            print(f"bins={binned_number}, data_strat={data_strat}")
            print(f"All 4 methods executed for this combination.")
            print(f"{'SUCCESS '*10}")
            
        except Exception as e:
            print(f"\\n{'ERROR '*10}")
            print(f"ERROR IN COMBINATION {combination_num}/{total_combinations}")
            print(f"bins={binned_number}, data_strat={data_strat}")
            print(f"Error: {str(e)}")
            print(f"{'ERROR '*10}")
            
            # Option to continue or stop on error
            import traceback
            print(f"\\nFull traceback:")
            traceback.print_exc()
            
            # Uncomment the next line if you want to stop on first error
            # raise e
            
            print(f"\\nContinuing to next combination...")

# ================================================================
# BATCH PROCESSING COMPLETED
# ================================================================
print(f"\\n{'='*100}")
print("BATCH PROCESSING COMPLETED!")
print("="*100)
print(f"Processed {len(bins)} bins × {len(data_strats)} data strategies = {len(bins) * len(data_strats)} combinations")
print(f"Each combination ran 4 methods (K-fold, Campaign-based, Day-based, Random-based)")
print("="*100)

# Summary of what was created
print(f"\\nSUMMARY OF GENERATED SPLITS:")
print(f"Main directory: MAC Machine Learning Model Code/split_datasets/")
print(f"")
for data_strat in data_strats:
    print(f"data_split_strat{data_strat}/")
    for binned_number in bins:
        if binned_number == 3.1:
            folder_name = "MAC_DATASET_LLOD_FILTERED"
        else:
            folder_name = f"MAC_binning_{binned_number}bins_optimized"
        print(f"  {folder_name}/")
        print(f"    kfold_based_split/     (Method 1: K-fold CV)")
        print(f"    campaign_based_split/  (Method 2: Campaign-based)")
        print(f"    day_based_split/       (Method 3: Day-based)")
        print(f"    random_based_split/    (Method 4: Random-based)")
        print(f"")

print("All data splitting combinations completed successfully!")
print("Ready for machine learning model training!")

SyntaxError: unmatched ')' (344481782.py, line 157)